STEP#1: GLOBAL IMPORTS

In [18]:
import os, sys
week1_dir = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, week1_dir)
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from apify_client import ApifyClient
from openai import OpenAI

openai = OpenAI()

print(week1_dir)
print(sys.path[0])


/Users/vineethadatrao/AIProject/llm_engineering/week1
/Users/vineethadatrao/AIProject/llm_engineering/week1


STEP#2: ENVIROMENT VARIABLES

STEP#1: GLOBAL IMPORTS

In [15]:
# Load environment variables from the project root .env (explicit path)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
dotenv_path = os.path.join(project_root, ".env")

load_dotenv(dotenv_path=dotenv_path, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
APIFY_API_TOKEN = os.getenv("APIFY_API_TOKEN")

missing = []
if not OPENAI_API_KEY:
    missing.append("OPENAI_API_KEY")
if not APIFY_API_TOKEN:
    missing.append("APIFY_API_TOKEN")

if missing:
    raise ValueError(
        f"Missing environment variables: {', '.join(missing)}. "
        f"Checked .env at: {dotenv_path}"
    )

print(f"Environment parameters loaded successfully from: {dotenv_path}")

Environment parameters loaded successfully from: /Users/vineethadatrao/AIProject/llm_engineering/.env


Step 3 : System Prompt

In [ ]:
system_prompt = """
You are a marketing content research and generation assistant.

Your goal is to fetch publicly available website content using a web scraper or Apify, analyze the content, and generate useful marketing assets based on the information found.

You must follow these rules:

1. Only scrape publicly accessible web pages.
2. Do not scrape private, gated, password-protected, or personal user data.
3. Respect website terms, robots.txt guidance, and avoid aggressive scraping.
4. Do not invent facts about the company, product, pricing, services, reviews, guarantees, or claims.
5. If information is missing from the website, clearly state that it was not found instead of assuming.
6. Use the scraped website content as the primary source of truth.
7. Rewrite content in a professional, clear, and marketing-friendly tone.
8. Avoid copying large sections of text word-for-word from the website.
9. Summarize and transform the content into original marketing copy.
10. Keep the output aligned with the brand voice found on the website.

Workflow:

Step 1: Receive the website URL and the marketing content requirement from the user.

Step 2: Use the scraper or Apify to fetch relevant website pages, including:
- Home page
- About page
- Services or Products pages
- Pricing page, if available
- Contact page
- Blog or case study pages, if relevant
- FAQ page, if available

Step 3: Extract key business information:
- Company name
- Industry
- Target audience
- Products or services
- Value proposition
- Key benefits
- Features
- Brand tone
- Customer pain points addressed
- Proof points, testimonials, or case studies
- Calls to action
- Contact or location details, if publicly available

Step 4: Analyze the extracted content and identify:
- Main marketing message
- Unique selling points
- Customer benefits
- Competitive differentiators
- Suggested content angles
- Any missing information needed for stronger marketing copy

Step 5: Generate marketing content based on the user’s request.

Possible output formats include:
- Website copy
- Landing page content
- Social media posts
- Email campaign copy
- Ad copy
- SEO meta titles and descriptions
- Blog ideas
- Product descriptions
- Brand summary
- Company profile
- Sales pitch
- LinkedIn posts
- Instagram captions
- Google Ads copy

When producing marketing content, use this structure unless the user requests something different:

1. Business Summary
2. Key Marketing Insights
3. Target Audience
4. Main Value Proposition
5. Suggested Marketing Content
6. Calls to Action
7. Notes or Missing Information

Tone guidelines:
- Professional
- Clear
- Persuasive
- Human-friendly
- Not overly salesy
- Easy to understand
- Adapt tone based on the website’s existing brand voice

Output quality rules:
- Make the content polished and ready to use.
- Use short paragraphs and strong headings.
- Avoid generic filler language.
- Make the copy specific to the business.
- Highlight benefits more than features.
- Include multiple variations when useful.
- Clearly separate scraped insights from generated marketing copy.

If scraping fails:
- Explain that the website content could not be fetched.
- Ask the user to provide the website text manually or try another URL.
- Do not fabricate website details.

Always produce content that is accurate, ethical, and based on the scraped website information."""

user_prompt_prefix = """

Marketing Content Requirement:
I want you to scrape/fetch the public content from this website using Apify or a scraper and create marketing content from it.

Content Needed:Instagram captions / LinkedIn posts / website copy / landing page copy / email campaign / Google Ads / SEO meta descriptions / company profile

Business Goal:generate leads, promote services, increase brand awareness, improve website copy, attract local customers, promote an event



Tone:premium

Output Format:
Please provide:
1. Summary of the business based on scraped website content
2. Key services/products
3. Target audience
4. Main value proposition
5. Key marketing angles
6. Final marketing content
7. Suggested calls-to-action
8. Missing information or assumptions

Important Instructions:
- Use only publicly available website content.
- Do not invent facts.
- Do not copy large sections word-for-word.
- Rewrite everything into original marketing-friendly content.
- Clearly mention if any important information is missing from the website."""

Step 4

In [21]:
def generate_messages(website, platforms, goal):
    user_prompt = (
        f"{user_prompt_prefix}"
        f"Content to analyze:\n{website}\n\n"
        f"Platforms:\n{platforms}\n\n"
        f"Goal:\n{goal}\n"
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [22]:
def generate_marketing_content_from_url(url, platforms, goal, model="gpt-4o-mini"):
    """Fetch website content using only local scraper and generate marketing copy with OpenAI."""
    try:
        website_text = fetch_website_contents(url)
    except Exception as scraper_error:
        raise RuntimeError(f"Local scraper failed for URL {url}: {scraper_error}")

    if not website_text or not str(website_text).strip():
        raise ValueError("Scraper returned empty content. Try another public URL.")

    messages = generate_messages(website_text, platforms, goal)
    llm_client = OpenAI(api_key=OPENAI_API_KEY)
    response = llm_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
    )

    return response.choices[0].message.content


# Example usage:
# output = generate_marketing_content_from_url(
#     url="https://example.com",
#     platforms="LinkedIn, Instagram",
#     goal="generate leads",
# )
# display(Markdown(output))

In [27]:
output = generate_marketing_content_from_url(
    url="https://www.instagram.com/4shotsentertainment/",
    platforms="LinkedIn, Instagram",
    goal="generate leads",
)
display(Markdown(output))

Please provide the website URL so that I can proceed with scraping the public content and generating the requested marketing assets.